# Grid Search: Valor do Clamp no Kernel Hiperbólico do HoTHP

**Objetivo:** Justificar empiricamente a escolha do valor de clamp usado nos expoentes
do kernel hiperbólico (`exp(-|Δt| · (θ' ± θⱼ))`).

**Hipótese:** O valor do clamp é uma guarda numérica, não um hiperparâmetro sensível.
Valores suficientemente grandes (≥10) não devem afetar o desempenho, pois o parâmetro
`time_scale` aprendido adapta a escala temporal antes do clamp atuar.

**Protocolo:**
- Grid: clamp ∈ {5, 10, 15, 20, 30, 50, 70, None (sem clamp)}
- 2 regimes temporais: escala natural (sintético) + escala extrema (×1000)
- 3 seeds, 50 épocas cada
- Métricas: NLL final, frequência de ativação do clamp, `time_scale` aprendido

**Run on Colab:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Instalacao e setup ────────────────────────────────────────────────
import os, sys

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git

sys.path.insert(0, 'ufc-easytpp')

!pip install omegaconf datasets pyyaml matplotlib pandas seaborn tqdm -q

# Fix: __init__.py minimo
_init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
with open(_init_path, 'w') as f:
    f.write(
        "from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel\n"
        "from easy_tpp.model.torch_model.torch_thp    import THP    as TorchTHP\n"
        "from easy_tpp.model.torch_model.torch_rothp  import RoTHP  as TorchRoTHP\n"
        "from easy_tpp.model.torch_model.torch_hothp  import HoTHP  as TorchHoTHP\n"
    )

# Fix: forca import de modelos no runner
_runner_path = 'ufc-easytpp/easy_tpp/runner/tpp_runner.py'
with open(_runner_path, 'r') as f:
    _content = f.read()
if 'import easy_tpp.model' not in _content:
    _content = _content.replace(
        'from collections import OrderedDict\n',
        'from collections import OrderedDict\nimport easy_tpp.model  # noqa: F401\n'
    )
    with open(_runner_path, 'w') as f:
        f.write(_content)

import torch
GPU = 0 if torch.cuda.is_available() else -1
print(f'OK — GPU={GPU}, torch={torch.__version__}', flush=True)

import json, time, gc, tempfile, copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml
%matplotlib inline

In [ ]:
# ── Preparar datasets: escala natural + escala extrema (×1000) ────────
from pathlib import Path

SRC_DIR = Path('ufc-easytpp/datasets/synthetic_hothp_scenarios/multi_scale_mixture')
SCALED_DIR = Path('./datasets/scaled_1000x')
SCALED_DIR.mkdir(parents=True, exist_ok=True)

SCALE_FACTOR = 1000.0

for split in ['train', 'validation', 'test']:
    with open(SRC_DIR / f'{split}.json') as f:
        data = json.load(f)
    scaled = []
    for seq in data:
        s = copy.deepcopy(seq)
        s['time_since_start'] = [t * SCALE_FACTOR for t in s['time_since_start']]
        s['time_since_last_event'] = [dt * SCALE_FACTOR for dt in s['time_since_last_event']]
        scaled.append(s)
    with open(SCALED_DIR / f'{split}.json', 'w') as f:
        json.dump(scaled, f)

# Verificar
with open(SCALED_DIR / 'train.json') as f:
    d = json.load(f)
dts = np.diff(d[0]['time_since_start'])
print(f'Escala natural:  mean_dt ~ 0.25', flush=True)
print(f'Escala ×{SCALE_FACTOR:.0f}: mean_dt ~ {np.mean([np.diff(s["time_since_start"]).mean() for s in d[:50]]):.1f}', flush=True)
print(f'Num event types: {d[0]["dim_process"]}', flush=True)

In [ ]:
# ── Configuracao do grid search ──────────────────────────────────────

CLAMP_VALUES = [5, 10, 15, 20, 30, 50, 70, None]  # None = sem clamp
SEEDS = [2019, 2020, 2021]
NUM_EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
NUM_HEADS = 2
NUM_LAYERS = 2
DROPOUT = 0.1
TIME_EMB_SIZE = 16
MAX_LEN = 50
NUM_EVENT_TYPES = 4  # synthetic datasets have dim_process=4

DATASETS = {
    'natural': {
        'train': str(SRC_DIR / 'train.json'),
        'valid': str(SRC_DIR / 'validation.json'),
        'test':  str(SRC_DIR / 'test.json'),
    },
    'scaled_1000x': {
        'train': str(SCALED_DIR / 'train.json'),
        'valid': str(SCALED_DIR / 'validation.json'),
        'test':  str(SCALED_DIR / 'test.json'),
    },
}

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Clamp values: {CLAMP_VALUES}', flush=True)
print(f'Datasets: {list(DATASETS.keys())}', flush=True)
print(f'Seeds: {SEEDS}', flush=True)
print(f'Total runs: {len(CLAMP_VALUES) * len(DATASETS) * len(SEEDS)}', flush=True)

In [ ]:
# ── Funcoes auxiliares ────────────────────────────────────────────────

from easy_tpp.config_factory import Config
from easy_tpp.runner import Runner


def make_config(clamp_val, dataset_name, seed, stage='train', pretrained_model_dir=None):
    clamp_label = 'none' if clamp_val is None else str(clamp_val)
    exp_id = f'clamp_{clamp_label}_{dataset_name}_s{seed}_{stage}'
    ds = DATASETS[dataset_name]

    model_cfg = {
        'hidden_size':    HIDDEN_SIZE,
        'num_heads':      NUM_HEADS,
        'num_layers':     NUM_LAYERS,
        'dropout':        DROPOUT,
        'time_emb_size':  TIME_EMB_SIZE,
        'use_ln':         False,
        'clamp_value':    clamp_val,
        'thinning': {
            'num_sample': 1, 'num_exp': 500, 'look_ahead_time': 10,
            'patience_counter': 5, 'over_sample_rate': 5,
            'num_samples_boundary': 5, 'dtime_max': 10,
            'num_seq': 10, 'num_step_gen': 1,
        },
    }
    if stage == 'train':
        model_cfg['loss_integral_num_sample_per_step'] = 20
        model_cfg['mc_num_sample_per_step'] = 20
    if pretrained_model_dir:
        model_cfg['pretrained_model_dir'] = pretrained_model_dir

    trainer_cfg = {
        'batch_size': BATCH_SIZE,
        'max_epoch':  NUM_EPOCHS if stage == 'train' else 1,
        'seed':       seed,
        'gpu':        GPU,
        'metrics':    ['acc', 'rmse'],
    }
    if stage == 'train':
        trainer_cfg.update({
            'valid_freq': 1, 'use_tfb': False,
            'optimizer': 'adam', 'learning_rate': LEARNING_RATE,
            'shuffle': False,
        })

    config_dict = {
        'pipeline_config_id': 'runner_config',
        'data': {
            dataset_name: {
                'data_format': 'json',
                'train_dir': ds['train'],
                'valid_dir': ds['valid'],
                'test_dir':  ds['test'],
                'data_specs': {
                    'num_event_types':    NUM_EVENT_TYPES,
                    'pad_token_id':       NUM_EVENT_TYPES,
                    'padding_side':       'right',
                    'truncation_side':    'right',
                    'truncation_strategy': 'longest_first',
                    'max_len':            MAX_LEN,
                },
            },
        },
        exp_id: {
            'base_config': {
                'stage':      stage,
                'backend':    'torch',
                'dataset_id': dataset_name,
                'runner_id':  'std_tpp',
                'model_id':   'HoTHP',
                'base_dir':   f'./checkpoints/clamp_grid/{clamp_label}/{dataset_name}/seed{seed}/',
            },
            'trainer_config': trainer_cfg,
            'model_config':  model_cfg,
        },
    }
    return config_dict, exp_id


def write_yaml_and_load(config_dict, experiment_id):
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(config_dict, f, default_flow_style=False)
        tmp_path = f.name
    try:
        cfg = Config.build_from_yaml_file(tmp_path, experiment_id=experiment_id)
    finally:
        os.unlink(tmp_path)
    return cfg


def get_model_dir(clamp_val, dataset_name, seed):
    clamp_label = 'none' if clamp_val is None else str(clamp_val)
    base = f'./checkpoints/clamp_grid/{clamp_label}/{dataset_name}/seed{seed}'
    if not os.path.isdir(base):
        return f'{base}/models/saved_model'
    candidates = []
    for entry in os.scandir(base):
        if entry.is_dir():
            candidate = os.path.join(entry.path, 'models', 'saved_model')
            if os.path.exists(candidate):
                candidates.append((entry.stat().st_mtime, candidate))
    if candidates:
        return sorted(candidates)[-1][1]
    return f'{base}/models/saved_model'


print('Funcoes auxiliares definidas.', flush=True)

In [ ]:
# ── Treinamento: grid search ─────────────────────────────────────────

results = []
total = len(CLAMP_VALUES) * len(DATASETS) * len(SEEDS)
run_idx = 0

for dataset_name in DATASETS:
    for clamp_val in CLAMP_VALUES:
        for seed in SEEDS:
            run_idx += 1
            clamp_label = 'None' if clamp_val is None else str(clamp_val)
            print(f'\n[{run_idx}/{total}] clamp={clamp_label}, '
                  f'dataset={dataset_name}, seed={seed}', flush=True)

            # ── Treino ──
            cfg_dict, exp_id = make_config(clamp_val, dataset_name, seed, stage='train')
            cfg = write_yaml_and_load(cfg_dict, experiment_id=exp_id)
            runner = Runner.build_from_config(cfg)

            t0 = time.time()
            try:
                runner.run()
                train_ok = True
            except Exception as e:
                print(f'  [FAIL] {e}', flush=True)
                train_ok = False
            elapsed = time.time() - t0

            # ── Captura time_scale aprendido ──
            time_scale_val = float('nan')
            if train_ok:
                try:
                    model = runner._model.model
                    time_scale_val = model.hope_emb.time_scale.item()
                except:
                    pass

            del runner
            free_gpu()

            if not train_ok:
                results.append({
                    'clamp': clamp_label, 'dataset': dataset_name, 'seed': seed,
                    'nll': float('nan'), 'acc': float('nan'), 'rmse': float('nan'),
                    'time_scale': float('nan'), 'train_time': elapsed, 'status': 'FAIL',
                })
                continue

            # ── Avaliacao ──
            model_dir = get_model_dir(clamp_val, dataset_name, seed)
            cfg_dict_eval, exp_id_eval = make_config(
                clamp_val, dataset_name, seed,
                stage='eval', pretrained_model_dir=model_dir)
            cfg_eval = write_yaml_and_load(cfg_dict_eval, experiment_id=exp_id_eval)
            runner_eval = Runner.build_from_config(cfg_eval)

            test_loader = runner_eval._data_loader.test_loader()
            metrics = runner_eval._evaluate_model(test_loader)
            del runner_eval
            free_gpu()

            nll = -metrics.get('loglike', float('nan'))
            acc = metrics.get('acc', float('nan'))
            rmse = metrics.get('rmse', float('nan'))

            results.append({
                'clamp': clamp_label, 'dataset': dataset_name, 'seed': seed,
                'nll': nll, 'acc': acc, 'rmse': rmse,
                'time_scale': time_scale_val, 'train_time': elapsed, 'status': 'OK',
            })
            print(f'  NLL={nll:.4f}  ACC={acc:.4f}  RMSE={rmse:.4f}  '
                  f'time_scale={time_scale_val:.6f}  ({elapsed:.0f}s)', flush=True)

print(f'\nGrid search concluido! {len(results)} resultados.', flush=True)

In [ ]:
# ── Tabela de resultados (media +/- std por clamp × dataset) ────────

df = pd.DataFrame(results)
df_ok = df[df['status'] == 'OK'].copy()

print('=' * 100, flush=True)
print('GRID SEARCH: CLAMP VALUE — HoTHP', flush=True)
print('=' * 100, flush=True)

for dataset_name in DATASETS:
    sub = df_ok[df_ok['dataset'] == dataset_name]
    print(f'\n--- Dataset: {dataset_name} ---', flush=True)
    print(f'{"clamp":>8} {"NLL":>20} {"ACC":>20} {"RMSE":>20} {"time_scale":>20}', flush=True)
    print('-' * 92, flush=True)

    for clamp_label in ['5', '10', '15', '20', '30', '50', '70', 'None']:
        cs = sub[sub['clamp'] == clamp_label]
        if len(cs) == 0:
            continue
        nll_str  = f'{cs["nll"].mean():.4f} +/- {cs["nll"].std():.4f}'
        acc_str  = f'{cs["acc"].mean():.4f} +/- {cs["acc"].std():.4f}'
        rmse_str = f'{cs["rmse"].mean():.4f} +/- {cs["rmse"].std():.4f}'
        ts_str   = f'{cs["time_scale"].mean():.6f} +/- {cs["time_scale"].std():.6f}'
        print(f'{clamp_label:>8} {nll_str:>20} {acc_str:>20} {rmse_str:>20} {ts_str:>20}', flush=True)

# Falhas
df_fail = df[df['status'] == 'FAIL']
if len(df_fail) > 0:
    print(f'\n*** {len(df_fail)} runs falharam (NaN/overflow): ***', flush=True)
    for _, row in df_fail.iterrows():
        print(f'  clamp={row["clamp"]}, dataset={row["dataset"]}, seed={row["seed"]}', flush=True)

In [ ]:
# ── Grafico principal: NLL vs clamp value ────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
COLORS = {'natural': '#2196F3', 'scaled_1000x': '#E53935'}

clamp_order = [5, 10, 15, 20, 30, 50, 70]
clamp_labels_ordered = [str(c) for c in clamp_order] + ['None']
x_pos = list(range(len(clamp_labels_ordered)))

for ax_idx, metric in enumerate(['nll', 'rmse']):
    ax = axes[ax_idx]
    for dataset_name, color in COLORS.items():
        means, stds = [], []
        valid_x = []
        for i, cl in enumerate(clamp_labels_ordered):
            sub = df_ok[(df_ok['dataset'] == dataset_name) & (df_ok['clamp'] == cl)]
            if len(sub) > 0:
                means.append(sub[metric].mean())
                stds.append(sub[metric].std())
                valid_x.append(i)
        if means:
            ax.errorbar(valid_x, means, yerr=stds, marker='o', capsize=4,
                       label=dataset_name, color=color, linewidth=2, markersize=6)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(clamp_labels_ordered)
    ax.set_xlabel('Clamp value')
    ax.set_ylabel(metric.upper())
    title = 'NLL (menor = melhor)' if metric == 'nll' else 'RMSE (menor = melhor)'
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Sensibilidade do HoTHP ao valor de clamp\n'
             '(3 seeds, 50 epocas, multi_scale_mixture)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('clamp_grid_search_nll_rmse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: clamp_grid_search_nll_rmse.png', flush=True)

In [ ]:
# ── Grafico: time_scale aprendido vs clamp ───────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))

for dataset_name, color in COLORS.items():
    means, stds = [], []
    valid_x = []
    for i, cl in enumerate(clamp_labels_ordered):
        sub = df_ok[(df_ok['dataset'] == dataset_name) & (df_ok['clamp'] == cl)]
        if len(sub) > 0 and sub['time_scale'].notna().any():
            means.append(sub['time_scale'].mean())
            stds.append(sub['time_scale'].std())
            valid_x.append(i)
    if means:
        ax.errorbar(valid_x, means, yerr=stds, marker='s', capsize=4,
                   label=dataset_name, color=color, linewidth=2, markersize=6)

ax.set_xticks(x_pos)
ax.set_xticklabels(clamp_labels_ordered)
ax.set_xlabel('Clamp value')
ax.set_ylabel('Learned time_scale')
ax.set_title('time_scale aprendido pelo HoTHP vs valor de clamp\n'
             '(escala natural: time_scale ~ 1.0, escala ×1000: time_scale << 1.0)',
             fontweight='bold')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='time_scale = 1.0')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('clamp_grid_search_time_scale.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: clamp_grid_search_time_scale.png', flush=True)

## Análise e Conclusões

**Preencher após execução:**

1. **Sensibilidade ao clamp (escala natural):**
   - Se NLL estável para clamp ≥ 10: o clamp não interfere quando a escala é pequena
   - O `time_scale` aprendido deve ficar próximo de 1.0

2. **Sensibilidade ao clamp (escala ×1000):**
   - Se NLL estável para clamp ≥ 10: o `time_scale` compensa a escala antes do clamp atuar
   - Se clamp=5 degrada: o clamp está interferindo no regime normal do kernel
   - Se `None` (sem clamp) falha: confirma necessidade da guarda numérica

3. **Escolha recomendada:**
   - Se resultados insensíveis para clamp ∈ [10, 70]: qualquer valor nessa faixa serve
   - 30 é justificável como ponto conservador no meio da faixa estável

4. **`time_scale` aprendido:**
   - Escala natural: espera-se ~1.0 (sem necessidade de reescalar)
   - Escala ×1000: espera-se ~0.001 (modelo aprende a inverter a escala)
   - Se consistente entre valores de clamp: o `time_scale` é o mecanismo primário, não o clamp